# 🎨 나만의 커스텀 퀵 드로우 만들기

오늘 우리는 **그림을 보고 무엇인지 알아맞히는 AI**를 직접 만들고,
**펜마우스로 그림을 그려서** 내가 만든 AI와 게임을 합니다.

순서는 이렇습니다.

1. 준비물 설치
2. 그림 데이터 가져오기
3. 데이터 살펴보기
4. 학습 준비
5. AI 두뇌(모델) 만들고 학습시키기
6. 얼마나 똑똑해졌는지 확인
7. 펜마우스로 그려서 맞히기! 🖊️

> 💡 **사용법**: 각 칸(셀)을 누르고 `Shift + Enter` 를 누르거나, 왼쪽의 ▶️ 버튼을 누르면 실행됩니다.
> 위에서부터 **순서대로** 실행하세요.
>
> ⚠️ **중요**: `4단계(학습 준비)`와 `5단계(모델 만들기)` 코드 셀에는 **TODO(빈칸/수정)** 이 들어있어요.
> 해당 부분을 채우고 실행해야 다음 단계로 넘어갑니다.

## 1단계 · 준비물 설치
펜마우스로 그릴 화면을 만들어 주는 `gradio` 를 설치합니다. (10~20초 걸려요)

In [ ]:
!pip -q install gradio
print("설치 완료!")

## 2단계 · 그림 데이터 가져오기
구글이 공개한 '낙서 데이터'를 내려받습니다.
아래 `categories` 목록을 **내 게임에 쓸 그림으로 바꿔보세요!** (3~5개 추천)

🔎 어떤 그림이 있는지 구경: https://quickdraw.withgoogle.com/data

In [ ]:
# ✏️ 여기를 내가 원하는 그림으로 바꿔보세요! (영어 이름, 3~5개 추천)
categories = ["cat", "apple", "house", "fish", "star"]

import os, urllib.request
os.makedirs("data", exist_ok=True)
base = "https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/"

for c in categories:
    path = f"data/{c}.npy"
    if not os.path.exists(path):
        print(f"내려받는 중: {c} ...")
        urllib.request.urlretrieve(base + c.replace(" ", "%20") + ".npy", path)
print("\n모든 그림 준비 완료! 🎉")

In [ ]:
import numpy as np

N_PER_CLASS = 5000  # ✏️ 그림 종류당 사용할 장 수 (늘리면 더 똑똑해지지만 더 느려요)

X, y = [], []
for label, c in enumerate(categories):
    data = np.load(f"data/{c}.npy")[:N_PER_CLASS]
    X.append(data)
    y += [label] * len(data)
    print(f"{c}: {len(data)}장")

X = np.concatenate(X).astype("float32")
y = np.array(y)
print("\n전체 그림 수:", len(X))

## 2.5단계 · (선택) 수집기 JSON 데이터 합치기
수업 초반에 사용한 **나만의 퀵드로우 수집기**에서 내려받은 JSON(`quickdraw_collector_dataset.json`)을
지금 데이터셋에 합칠 수 있어요.

- `USE_COLLECTOR_DATA = True` 로 바꾸고 셀을 실행하면 업로드 창이 열립니다.
- 업로드한 `objectName` 이 `categories` 안에 있으면 해당 클래스에 데이터를 추가합니다.
- 없으면 새 클래스로 자동 추가합니다. (이 경우 클래스 수가 늘어납니다)

> 기본값은 `False` 이며, 그대로 두면 기존 수업과 완전히 동일하게 진행됩니다.

In [ ]:
# ✏️ 수집기 JSON을 합치려면 True로 바꾸세요.
USE_COLLECTOR_DATA = False

if USE_COLLECTOR_DATA:
    import json

    try:
        from google.colab import files
        uploaded = files.upload()
    except Exception:
        uploaded = {}
        print("Colab 환경이 아니거나 업로드 창을 열 수 없습니다. 이 단계는 건너뜁니다.")

    if uploaded:
        fname, raw = next(iter(uploaded.items()))
        payload = json.loads(raw.decode("utf-8"))

        collector_label = str(payload.get("objectName", "collector")).strip().lower()
        samples = payload.get("samples", [])

        vecs = []
        for s in samples:
            v = np.array(s.get("vec", []), dtype="float32")
            if v.size == 784:
                vecs.append(v)

        if vecs:
            collector_data = (np.stack(vecs) * 255.0).astype("float32")

            if collector_label in categories:
                label_idx = categories.index(collector_label)
            else:
                categories.append(collector_label)
                label_idx = len(categories) - 1

            X = np.concatenate([X, collector_data], axis=0)
            y = np.concatenate([y, np.full(len(collector_data), label_idx, dtype=int)], axis=0)

            print(f"수집기 파일: {fname}")
            print(f"추가된 라벨: {collector_label} (index={label_idx})")
            print(f"추가된 샘플 수: {len(collector_data)}")
            print("업데이트된 categories:", categories)
            print("업데이트된 전체 그림 수:", len(X))
        else:
            print("유효한 샘플(vec 길이 784)을 찾지 못했습니다.")
    else:
        print("업로드된 파일이 없습니다. 이 단계를 건너뜁니다.")
else:
    print("수집기 데이터 합치기: 건너뜀 (USE_COLLECTOR_DATA=False)")

## 3단계 · 데이터 살펴보기
컴퓨터가 학습할 그림들이 어떻게 생겼는지 눈으로 확인해 봅시다.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    idx = np.random.randint(len(X))
    plt.imshow(X[idx].reshape(28, 28), cmap="gray")
    plt.title(categories[y[idx]])
    plt.axis("off")
plt.show()

## 4단계 · 학습 준비 (여기서부터는 “코드 작성”)
그림 데이터를 0~1로 정규화하고, CNN이 읽을 수 있는 형태(28×28×채널)로 바꿉니다.

> 다음 코드 셀에서 **DIVISOR / CHANNELS** 는 직접 채워야 실행됩니다.
> (예: `DIVISOR = 255`, `CHANNELS = 1`)

In [ ]:
from sklearn.model_selection import train_test_split

# TODO: 아래 값을 직접 채워보세요.
# - DIVISOR: 0~255 → 0~1 로 만들기 위한 값 (예: 255)
# - CHANNELS: 흑백 이미지는 보통 1
DIVISOR = None
CHANNELS = None

if DIVISOR is None or CHANNELS is None:
    raise ValueError(
        "DIVISOR / CHANNELS 를 채워주세요. 예: DIVISOR=255, CHANNELS=1"
    )

X = X / DIVISOR
X = X.reshape(-1, 28, 28, CHANNELS)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("학습용:", len(X_train), "장   시험용:", len(X_test), "장")

## 5단계 · AI 두뇌(모델) 만들고 학습시키기 (여기서도 “코드 작성”)
AI 모델(CNN)을 만들고 학습시킵니다.

> 다음 코드 셀에서 **ACTIVATION / FILTERS** 를 지정하고,
> `add_conv_block()` 의 **TODO를 완성해야 실행됩니다.**

완성 후 실행하면, `accuracy`가 올라가는 과정을 볼 수 있어요!

### ✍️ 코드 작성 체크리스트 (4~5단계)
아래 순서대로 직접 채우고 실행해보세요.

1. **4단계 코드 셀**
   - `DIVISOR = 255`
   - `CHANNELS = 1`
2. **5단계 코드 셀**
   - `ACTIVATION = "relu"`
   - `FILTERS = [32, 64]`
3. `add_conv_block()` 함수의 TODO를 완성
   - `Conv2D`
   - `MaxPooling2D`
4. 실행 후 정확도(`accuracy`)가 어떻게 변하는지 확인

> 도전: `FILTERS = [16, 32]` 또는 `FILTERS = [64, 128]`로 바꿔서 속도/정확도를 비교해보세요.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# TODO: 아래 값을 직접 채워보세요.
# - ACTIVATION: 예) "relu"
# - FILTERS: 예) [32, 64]
ACTIVATION = None
FILTERS = None

if ACTIVATION is None or FILTERS is None:
    raise ValueError(
        "ACTIVATION / FILTERS 를 채워주세요. 예: ACTIVATION='relu', FILTERS=[32, 64]"
    )

def add_conv_block(m, filters):
    """CNN에서 Conv2D → MaxPooling 을 1세트로 추가합니다."""
    # TODO: 아래를 완성하세요.
    # 1) Conv2D 추가
    # 2) MaxPooling2D 추가
    # 아래 예시를 참고해서 직접 코드를 작성해보세요!
    # 예시:
    # m.add(layers.Conv2D(filters, 3, activation=ACTIVATION))
    # m.add(layers.MaxPooling2D())
    raise NotImplementedError("add_conv_block() 의 TODO를 완성해주세요.")

model = models.Sequential()
model.add(layers.Input((28, 28, 1)))

for f in FILTERS:
    add_conv_block(model, f)

model.add(layers.Flatten())
model.add(layers.Dense(128, activation=ACTIVATION))
model.add(layers.Dense(len(categories), activation="softmax"))

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

EPOCHS = 5  # ✏️ 학습 반복 횟수 (늘리면 보통 더 똑똑해져요)
history = model.fit(
    X_train,
    y_train,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=128,
)

## 6단계 · 얼마나 똑똑해졌나 확인
한 번도 본 적 없는 '시험용' 그림으로 성적을 매겨봅니다.

In [ ]:
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"✅ 시험 성적(정확도): {acc * 100:.1f}%")

plt.plot(history.history["accuracy"], label="학습")
plt.plot(history.history["val_accuracy"], label="검증")
plt.xlabel("epoch"); plt.ylabel("정확도"); plt.legend(); plt.show()

## 7단계 · 펜마우스로 그려서 맞히기! 🖊️
이제 진짜 게임 시간! 아래를 실행하면 **그림판 링크**가 나옵니다.
펜마우스로 그림을 그리고 **Submit** 을 누르면, 내가 만든 AI가 맞혀줍니다.

> 잘 못 맞히면? 그림을 더 크고 또렷하게, 화면 가운데에 그려보세요.

In [ ]:
import os
import gradio as gr
from PIL import Image

# ===== (선택) 미드저니 아트 연동 =====
# True로 바꾸면, Colab에서 이미지를 업로드할 수 있어요.
# 파일 이름 규칙 예시:
# - cat.png 처럼 categories의 라벨과 같은 이름(소문자)을 쓰면 매칭됩니다.
USE_MIDJOURNEY_ART = False

ART_IMAGES = {}  # 예: {"cat": "cat.png"}

if USE_MIDJOURNEY_ART:
    try:
        from google.colab import files
        uploaded = files.upload()
        for fname in uploaded.keys():
            key = os.path.splitext(fname)[0].strip().lower()
            ART_IMAGES[key] = fname
        print("업로드된 아트 파일:", ART_IMAGES)
    except Exception as e:
        print("이미지 업로드를 사용할 수 없습니다:", e)


def load_art_for_label(label):
    if not label:
        return None
    path = ART_IMAGES.get(str(label).strip().lower())
    if not path:
        return None
    if not os.path.exists(path):
        return None
    try:
        img = Image.open(path).convert("RGB")
        # 너무 큰 이미지는 UI가 느려질 수 있어서 축소
        return img.resize((224, 224))
    except Exception:
        return None


def predict(sketch):
    if sketch is None:
        return {}, None

    # Gradio 버전에 따라 dict 또는 이미지로 들어옵니다
    if isinstance(sketch, dict):
        sketch = sketch.get("composite") or sketch.get("image") or sketch.get("background")

    img = np.array(sketch).astype("float32")

    # '그린 선'을 밝게 만들기 (학습 데이터는 검은 배경 + 흰 선)
    if img.ndim == 2:
        ink = 255 - img
    elif img.shape[2] == 4:  # RGBA
        alpha = img[..., 3]
        rgb = img[..., :3].mean(axis=2)
        ink = (alpha > 10) * (255 - rgb)
    else:  # RGB
        ink = 255 - img[..., :3].mean(axis=2)

    ink = Image.fromarray(ink.astype("uint8")).resize((28, 28))
    arr = np.array(ink).astype("float32") / 255.0
    arr = arr.reshape(1, 28, 28, 1)

    probs = model.predict(arr, verbose=0)[0]
    probs_dict = {categories[i]: float(probs[i]) for i in range(len(categories))}

    # 상위 1개 라벨의 아트(선택)를 함께 보여줌
    top_i = int(np.argmax(probs))
    top_label = categories[top_i]
    art_img = load_art_for_label(top_label)

    return probs_dict, art_img

demo = gr.Interface(
    fn=predict,
    inputs=gr.Sketchpad(),
    outputs=[
        gr.Label(num_top_classes=3, label="AI 예측"),
        gr.Image(type="pil", label="정답 아트 (선택)"),
    ],
    title="🎨 나만의 퀵 드로우",
    description="펜마우스로 그림을 그리고 Submit 을 눌러보세요!",
)
demo.launch(share=True)

## 🚀 더 해보기 (도전과제)
- `categories` 의 그림을 바꿔서 **나만의 테마** 게임 만들기 (예: 동물 게임, 음식 게임)
- `N_PER_CLASS` 나 `EPOCHS` 숫자를 키워서 정확도 올려보기
- (선택) 미드저니 아트를 게임에 연결하기
  - 위 Gradio 셀에서 `USE_MIDJOURNEY_ART = True` 로 바꾼 뒤,
  - Colab에서 이미지를 업로드하고 파일 이름을 `categories` 라벨과 맞추면
  - AI가 정답으로 예측할 때 해당 아트가 함께 보입니다.
- 미드저니로 **같은 스타일을 유지**해보세요: 예) `--style`, `--ar`, 색감/선 두께를 고정해서 3~5개 클래스에 공통 룩을 주기
- 데이터 편향 줄이기 팁: 같은 클래스라도 **서로 다른 각도/구도**의 아트를 여러 장 만들면, 학생들이 그릴 때도 다양성이 늘어나요.
- 친구에게 그림판 링크를 보내서 같이 플레이하기

**오늘의 질문**: AI가 잘 못 맞히는 그림은 무엇이었나요? 왜 그럴까요? 🤔